# Sharded Pipeline Quick Reference

| Metadata | Value |
|----------|-------|
| **Level** | Intermediate |
| **Runtime** | ~5 min |
| **Prerequisites** | Basic Datarax pipeline, JAX sharding concepts |
| **Format** | Python + Jupyter |

## Overview

Distribute a Datarax pipeline's batches across JAX devices. Give the pipeline the sharding of a
batch's rows and iteration places every batch on the device mesh, so the pipeline's operators
and your step run on each device's share of the batch.

## Learning Goals

By the end of this example, you will be able to:

1. Create a JAX device mesh for data parallelism
2. Build a pipeline that places its batches on the mesh
3. Verify how each batch is distributed across devices
4. Read an evaluation's short final batch, placed whole on every device

## Setup

```bash
# Install datarax
uv pip install datarax
```

The same code runs on one device (a mesh of one) and on several.

In [ ]:
# Imports
import jax
import numpy as np
from flax import nnx
from substrax.mesh import DeviceMeshManager
from substrax.spmd import create_data_parallel_sharding

from datarax.operators import ElementOperator, ElementOperatorConfig
from datarax.pipeline import Pipeline
from datarax.sources import MemorySource, MemorySourceConfig


print(f"JAX devices: {jax.devices()}")
print(f"Device count: {len(jax.devices())}")

## Step 1: Device Mesh Setup

A JAX `Mesh` organizes devices along named axes. substrax's `DeviceMeshManager` builds one with
Auto axes, so XLA infers how a computation over sharded inputs is partitioned. Common patterns:

- `("data",)` - Data parallelism across all devices
- `("data", "model")` - 2D mesh for data + model parallelism

In [ ]:
# Data parallelism: every device along the "data" axis
mesh = DeviceMeshManager.create_data_parallel_mesh()
sharding = create_data_parallel_sharding(mesh)
num_devices = mesh.devices.size

print(f"Mesh with {num_devices} device(s) along 'data'")
print(f"Batch sharding: {sharding.spec}")

## Step 2: Create Data and a Sharded Pipeline

The source and the operators are defined as for one device. The pipeline takes the sharding;
its batch size must split over the `"data"` axis, so it scales with the device count.

In [ ]:
# Create sample data
num_samples = 1024
rng = np.random.default_rng(0)
data = {
    "image": rng.random((num_samples, 32, 32, 3), dtype=np.float32),
    "feature": rng.random((num_samples, 128), dtype=np.float32),
    "label": rng.integers(0, 10, (num_samples,), dtype=np.int32),
}
source = MemorySource(MemorySourceConfig(), data=data)


# Define normalization operator
def normalize(element, key=None):
    """Normalize image to [0, 1] range."""
    return element.update_data({"image": element.data["image"] / 255.0})


normalizer = ElementOperator(
    ElementOperatorConfig(stochastic=False), fn=normalize, rngs=nnx.Rngs(0)
)

batch_size = 32 * num_devices  # 32 records per device
pipeline = Pipeline(
    source=source,
    stages=[normalizer],
    batch_size=batch_size,
    rngs=nnx.Rngs(0),
    sharding=sharding,
)

print(f"Data samples: {num_samples}")
print(f"Pipeline created with batch_size={batch_size}")

## Step 3: Process Sharded Batches

Each batch arrives with every array split along the `"data"` axis, and the pipeline's operators
ran on each device's share. The loop runs under `jax.set_mesh(mesh)`, where computations on the
sharded arrays run across the devices and APIs taking a sharding also accept a bare
`PartitionSpec`.

In [ ]:
# Process batches
print("\nProcessing batches:")
with jax.set_mesh(mesh):
    for i, batch in enumerate(pipeline):
        if i >= 2:
            break
        print(f"Batch {i}:")
        print(f"  Image shape: {batch['image'].shape}")
        print(f"  Image sharding: {batch['image'].sharding.spec}")
        print(f"  Rows per device: {batch['image'].addressable_shards[0].data.shape[0]}")
        print(f"  Label shape: {batch['label'].shape}")

# Expected output (2 devices):
# Batch 0:
#   Image shape: (64, 32, 32, 3)
#   Image sharding: P('data',)
#   Rows per device: 32
#   Label shape: (64,)

## Step 4: The Final Batch of an Evaluation

An evaluation over 1,000 records ends on a short batch, whose rows the `"data"` axis cannot
split. The pipeline places it whole on every device (`PartitionSpec()`), so every record is
evaluated once and no batch holds padding. A step over it compiles once more, on the first pass.

In [ ]:
evaluation = Pipeline(
    source=MemorySource(MemorySourceConfig(), data={k: v[:1000] for k, v in data.items()}),
    stages=[normalizer],
    batch_size=batch_size,
    rngs=nnx.Rngs(0),
    sharding=sharding,
)
with jax.set_mesh(mesh):
    batches = list(evaluation)
records = sum(batch["label"].shape[0] for batch in batches)
final = batches[-1]
print(f"Batches: {len(batches)}, records evaluated: {records}")
print(f"Final batch: {final['label'].shape[0]} rows, sharding {final['label'].sharding.spec}")

# Expected output (2 devices):
# Batches: 16, records evaluated: 1000
# Final batch: 40 rows, sharding P()

## Results Summary

| Feature | Value |
|---------|-------|
| Device Count | Depends on system |
| Mesh Shape | (N,) for N devices |
| Full batches | Rows split along `"data"` |
| An evaluation's final short batch | Whole on every device |

Sharding benefits:

- **Memory efficiency**: each device holds its share of every full batch
- **Throughput**: the pipeline's operators run on every device's share in parallel
- **Scalability**: the same code runs on one device or many

## Next Steps

- **Advanced sharding**: Explore model parallelism for large models
- **TPU sharding**: Configure meshes for TPU pod slices
- **Pipeline parallelism**: Overlap data loading and computation
- **Checkpointing**: [Checkpointing](../checkpointing/01_checkpoint_quickref.ipynb)

In [ ]:
def main():
    """Run the sharded pipeline example."""
    print("Sharded Pipeline Example")
    print("=" * 50)
    mesh = DeviceMeshManager.create_data_parallel_mesh()
    num_devices = mesh.devices.size
    rng = np.random.default_rng(0)
    data = {
        "image": rng.random((1024, 32, 32, 3), dtype=np.float32),
        "label": rng.integers(0, 10, (1024,), dtype=np.int32),
    }
    pipeline = Pipeline(
        source=MemorySource(MemorySourceConfig(), data=data),
        stages=[
            ElementOperator(ElementOperatorConfig(stochastic=False), fn=normalize, rngs=nnx.Rngs(0))
        ],
        batch_size=32 * num_devices,
        rngs=nnx.Rngs(0),
        sharding=create_data_parallel_sharding(mesh),
    )
    total_samples = 0
    with jax.set_mesh(mesh):
        for i, batch in enumerate(pipeline):
            if i >= 5:
                break
            total_samples += batch["image"].shape[0]
    print(f"Devices: {num_devices}, processed {total_samples} samples")
    print("Example completed successfully!")


if __name__ == "__main__":
    main()